In [11]:
%pip uninstall drjit
%pip install drjit
%pip show drjit

^C
Note: you may need to restart the kernel to use updated packages.
^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: C:\Users\shubi\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


Name: drjit
Version: 0.4.6
Summary: A Just-In-Time-Compiler for Differentiable Rendering
Home-page: https://github.com/mitsuba-renderer/drjit
Author: Wenzel Jakob
Author-email: wenzel.jakob@epfl.ch
License: BSD
Location: c:\users\shubi\appdata\local\programs\python\python310\lib\site-packages
Requires: 
Required-by: mitsuba
Note: you may need to restart the kernel to use updated packages.


In [13]:
%pip install mitsuba


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: mitsuba in c:\users\shubi\appdata\local\programs\python\python310\lib\site-packages (3.5.2)




[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: C:\Users\shubi\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [14]:
%pip install sionna

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: C:\Users\shubi\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [12]:
%pip install wandb

     ---------------------------------------- 0.0/20.2 MB ? eta -:--:--
     ---------------------------------------- 0.2/20.2 MB 4.6 MB/s eta 0:00:05
     - -------------------------------------- 0.7/20.2 MB 8.5 MB/s eta 0:00:03
     --- ------------------------------------ 1.7/20.2 MB 13.5 MB/s eta 0:00:02
     ------ --------------------------------- 3.2/20.2 MB 18.5 MB/s eta 0:00:01
     --------- ------------------------------ 4.6/20.2 MB 19.8 MB/s eta 0:00:01
     ------------ --------------------------- 6.1/20.2 MB 21.6 MB/s eta 0:00:01
     --------------- ------------------------ 7.6/20.2 MB 24.4 MB/s eta 0:00:01
     ----------------- ---------------------- 9.1/20.2 MB 25.3 MB/s eta 0:00:01
     -------------------- ------------------ 10.5/20.2 MB 28.5 MB/s eta 0:00:01
     ----------------------- --------------- 12.3/20.2 MB 31.2 MB/s eta 0:00:01
     -------------------------- ------------ 14.0/20.2 MB 32.7 MB/s eta 0:00:01
     ---------------------------- ---------- 14.9/


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: C:\Users\shubi\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [1]:
# Jason's implementation of directional algo
from algorithim import Params, Algorithm, Capon
from scipy.signal import find_peaks

import sionna
import tensorflow as tf

from IPython.core.magic import register_cell_magic
from IPython import get_ipython

In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import time

# Import Sionna RT components
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, Camera, AntennaArray, Antenna

# For link-level simulations
from sionna.channel import cir_to_ofdm_channel, subcarrier_frequencies, OFDMChannel, ApplyOFDMChannel, CIRDataset
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.utils import compute_ber, ebnodb2no, PlotBER
from sionna.ofdm import KBestDetector, LinearDetector
from sionna.mimo import StreamManagement
from sionna.ofdm import ResourceGrid

def calculate_heading_rad(start, end):
    # Calculate the difference vector
    diff = np.array(end) - np.array(start)
    
    # Calculate the heading angle in radians
    heading = np.arctan2(diff[1], diff[0])
    
    return heading

def get_rss_from_csi(csi):
  # Compute CSI power (power per subcarrier)
  csi_power = tf.abs(csi)**2

  # Compute RSS per receiver-transmitter pair (sum over subcarriers)
  rss_per_rx_tx = tf.reduce_sum(csi_power, axis=-1)  # Sum over subcarriers

  # Convert RSS from linear scale (e.g., Watts) to dBm
  rss_per_rx_tx_dBm = 10 * tf.math.log(rss_per_rx_tx / 1e-3) / tf.math.log(10.0)

  return rss_per_rx_tx_dBm

def dbm_to_watts(dbm):
    return 10. ** ((dbm-30)/10)

def watts_to_dbm(watts):
    epsilon = 1e-10  # Small constant to avoid log(0)
    return 10 * np.log10(watts + epsilon) + 30

# # Multiple Antennas like ASUS Router
def get_antenna_positions(spacing):
    return np.array([[0.0, spacing / 2 + spacing , 0.0],
                     [0.0, spacing / 2, 0.0],
                     [0.0, -spacing / 2, 0.0],
                     [0.0, -spacing / 2 - spacing, 0.0]])

def flip_trajectory(data):
    # Vector from origin to the first and last points of each trajectory
    first_points = data[:, 0, :]  # Shape (1000, 3)
    last_points = data[:, -1, :]  # Shape (1000, 3)

    # Compute the dot product of the first and last points with respect to the origin
    dot_products = np.sum(first_points * last_points, axis=1)

    # Identify trajectories heading away from the origin (dot product > 0)
    away_from_origin = dot_products < 0

    # Flip the trajectories that are not heading away from the origin
    data[~away_from_origin] = data[~away_from_origin, ::-1, :]

    return data

In [3]:
trajectories = np.load('trajectories_1000.npy')
print(trajectories.shape)

(500, 20, 3)


In [9]:
def max_lidar_distance(points, loc, max_dist):
    updated_PC = []
    for i in range(len(points)):
        if (np.linalg.norm(loc - points[i]) <= max_dist):
            updated_PC.append(points[i])
        else:
            updated_PC.append(np.array([float('inf'), float('inf'), float('inf')]))
    return np.array(updated_PC)    

def normalize_pc(points):
    # data that does not include all the infinites (just for calculating the mean and furthest_distance)
    cleaned_data = np.array(points[np.where(points[:, 0] != float('inf'))], dtype="float32")
    if (len(cleaned_data) == 0):
        return points

    centroid = np.mean(cleaned_data, axis=0)
    points -= centroid
    furthest_distance = np.max(np.sqrt(np.sum(abs(cleaned_data)**2,axis=-1)))
    points /= furthest_distance
    
    return np.array(points, dtype=np.float32)

lidar_load = np.load("lidar.npy", allow_pickle=True)

# # # an example
print(lidar_load.shape)
# print(lidar[0][0])
num_trajectories = lidar_load.shape[0]
num_steps = lidar_load.shape[1]
num_lidar = lidar_load.shape[2]

lidar = np.empty((0, num_lidar, 3))

for i in range(num_trajectories):
    for j in range(num_steps - 1):
        # reshape so that the output of normalize pc is one 
        lidar = np.concatenate((lidar, normalize_pc(max_lidar_distance(lidar_load[i][j], trajectories[i][j], 10)).reshape(1, num_lidar, 3)), axis=0)

print(lidar.shape)
lidar = lidar.reshape(num_trajectories, num_steps - 1, num_lidar, 3)
print(lidar.shape)

(500, 20, 100, 3)
(9500, 100, 3)
(500, 19, 100, 3)


In [15]:
lidar[1][0]

array([[            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf,             inf],
       [            inf,             inf

In [16]:
# trajectories = trajectories[:, ::5, :]
# print(trajectories.shape)
total_traj = trajectories.shape[0]
seq_len = 20
# data = np.empty(shape=(0, seq_len, 11), dtype=object)

print(total_traj)

500


In [62]:
data = np.empty((0, 310))

for traj in range(total_traj):
    # List for dataset
    angles = []
    angle_profile_values = []
    rssi = []
    rssi_normalized = []

    SCENE_NAME = "lunar_ex" #"canyon"
    IMAGE_FOLDER = f"images/{SCENE_NAME}"

    # Load integrated scene
    scene = load_scene(f"models/{SCENE_NAME}.xml") # Try also sionna.rt.scene.etoile

    # Configure antenna array for all transmitters
    scene.tx_array = PlanarArray(num_rows=1,
                                num_cols=1,
                                vertical_spacing=0.5,
                                horizontal_spacing=0.5,
                                pattern="iso",
                                polarization="V")

    antenna_positions = get_antenna_positions(0.03)
    antenna_array_angle = -175 # Default

    RelativeAntennas = AntennaArray(antenna=Antenna("dipole", "V"), 
                                positions=tf.Variable(antenna_positions.tolist()))

    # scene.tx_array = RelativeAntennas 
    scene.rx_array = RelativeAntennas

    # Create transmitter
    tx_position = [0., 1., 0.]
    tx = Transmitter(name="tx",
                    # position=[0., -10., 0.],
                    position=tx_position,
                    orientation=[np.radians(0),0,0])

    # Add transmitter instance to scene
    scene.add(tx)

    # trajectories = flip_trajectory(trajectories)
    trajectories = trajectories[:total_traj,:,:]
    trajectory_index = traj
    num_steps = seq_len
    trajectory_steps =  np.arange(num_steps - 1)

    # start_color = np.array([1, 0, 0])  # Red
    # end_color = np.array([0, 0, 1]) 
    # colors = [start_color + (end_color - start_color) * i / (num_steps - 1) for i in range(num_steps)]

    # Calculate headings for the trajectory
    calculated_headings = []
    for i in trajectory_steps: #range(0, len(trajectories[0]) - 1):
        heading = calculate_heading_rad(trajectories[trajectory_index,i,:2], trajectories[trajectory_index,i + 1,:2])
        calculated_headings.append(heading)

    rxs = []
    for i, position in enumerate(trajectories[trajectory_index, np.arange(num_steps) ,:]):
        if i == (seq_len - 1):
            heading = calculated_headings[-1]
        else:
            heading = calculated_headings[i]
        rxs.append(Receiver(name=f"rx{i}",
                position=position,
                orientation=[heading,0,0]))   
        scene.add(rxs[i])
        # if trajectory_index == 0:
        #     print(f"Receiver {i} position: {position}")

    antenna_array_angle = np.radians(180) # heading

    scene.frequency = 2.14e9 #5.745 # in Hz; implicitly updates RadioMaterials

    scene.synthetic_array = False # If set to False, ray tracing will be done per antenna element (slower for large arrays)

    # Compute propagation paths
    paths = scene.compute_paths(max_depth=2,
                                num_samples=1e5,  # Number of rays shot into directions defined
                                                # by a Fibonacci sphere , too few rays can
                                                # lead to missing paths
                                los=True,  # Include Line-of-Sight paths
                                reflection=True,  # Include reflection paths
                                diffraction=True,  # Include diffraction paths
                                scattering=True,  # Include scattering paths
                                )
    
    paths.normalize_delays = False

    # Determine subcarrier frequencies
    rg = ResourceGrid(num_ofdm_symbols=1,
                    fft_size=52,
                    dc_null = True,
                    cyclic_prefix_length=20,
                    #   pilot_pattern = "kronecker",
                    #   pilot_ofdm_symbol_indices = [2, 8],
                    subcarrier_spacing=5e6) #30e3)

    frequencies = subcarrier_frequencies(rg.fft_size, rg.subcarrier_spacing)

    # get rss
    a, tau = paths.cir()

    csi = cir_to_ofdm_channel(frequencies, a, tau, normalize=False)  # Non-normalized includes path-loss
   
    csi_reshaped = []
    for i in range(len(rxs)):
        csi_reshaped.append(np.array(csi[:,i,:,:,:,:,:]).reshape(4, 52))

    #Output: h_f ([batch size, num_rx, num_rx_ant, num_tx, num_tx_ant, num_time_steps, fft_size], tf.complex) – Channel frequency responses at frequencies
    block_rss = get_rss_from_csi(csi).numpy()[0, :, 0, 0, 0, 0]
    block_rss[np.isneginf(block_rss)] = -90
    block_rss[block_rss < -90] = -90 # Set a floor for RSS values to avoid extreme values
    rssi.append(block_rss[:-1])
    # Define range
    min_rss = -90
    max_rss = 0

    # Normalize values to 0-1
    normalized = (block_rss - min_rss) / (max_rss - min_rss)
    # print(f"normalized: {normalized} block_rrsi: {block_rss} min_rss: {min_rss} max_rss: {max_rss}")
    rssi_normalized.append(normalized[:-1])

    # # Define three colors
    # color_black = np.array([0, 0, 0])
    # color_red = np.array([1, 0, 0])
    # color_blue = np.array([0, 0, 1])
    # colors = []
    # for value in normalized:
    #     if value <= 0.5:
    #         # Scale between black and gray
    #         color = color_black + (color_red - color_black) * (value / 0.5)
    #     else:
    #         # Scale between gray and white
    #         color = color_red + (color_blue - color_red) * ((value - 0.5) / 0.5)
    #     colors.append(color)

    # Map normalized values to colors
    # 0-1 range is split into 3 intervals: [0, 1/3), [1/3, 2/3), [2/3, 1]
    # color_indices = np.digitize(normalized, bins=[1/3, 2/3], right=False)
    # mapped_colors = [colors[idx] for idx in color_indices]

    # for i, rx in enumerate(rxs):
    #     rx.color = colors[i]

    # Implementing Capon Algorithim
    params = Params(antenna_positions[:, :2])

    angle_tmp = []
    angle_profile_values_tmp = []
    for i in range(0, len(rxs) - 1):
        
        csi_tmp = tf.reshape(csi[:,i,:,:,:,:,:], [4, 52]) #  tf.reshape(csi[:,i,:,:,:,:,:], [1, -1])

        if np.all(csi_tmp.numpy() == 0):
            angle_tmp.append(np.zeros(3))
            angle_profile_values_tmp.append(np.zeros(3))
            continue
        
        AOA, theta_samples, profile_norm = Capon(params, 155, 80, csi_tmp).evaluate()

        # This for when going towards transmitter
        profile_norm = np.roll(profile_norm, 180)
        # Capture three top peaks from bottom of profile (90-270 degrees)
        evaluation = np.array([profile_norm[90:270]])
        evaluation = evaluation.flatten() 
        peaks, _ = find_peaks(evaluation)
        peak_values = evaluation[peaks]
        angle_profile_values_tmp.append(np.pad(peak_values, (0, max(0, 3 - peak_values.shape[0])), mode='constant'))
        angle_tmp.append(np.pad(theta_samples[90:270][peaks], (0, max(0, 3 - theta_samples[90:270][peaks].shape[0])), mode='constant'))
        AOA = np.array([AOA])


    angles.append(angle_tmp)
    angle_profile_values.append(angle_profile_values_tmp)

    ### Saving CSI
    csi_reshaped = np.array(csi_reshaped)

    # need this for amplitude to dB
    def dB(x):
        return 10 * np.log10(np.abs(x) ** 2)
        # return 10 * np.log10(np.maximum(np.abs(x) ** 2, 1e-8))

    # lets say we only care about Rx antenna #0
    C = dB(csi_reshaped[:, 0, :])

    # csi: shape (N_subcarriers, N_rx, N_tx), complex-valued
    # |H|^2 gives power per subcarrier
    csi_magnitude_squared = np.abs(csi_reshaped[0,:,:])**2
    power_per_subcarrier = csi_magnitude_squared.mean(axis=(0))  # average over Tx/Rx
    avg_csi_power_db = 10 * np.log10(np.mean(power_per_subcarrier))

    for i in range(0, len(rssi[0])):
        row = np.array([rssi_normalized[0][i], angles[0][i][0], angles[0][i][1], angles[0][i][2],
                                    angle_profile_values[0][i][0], angle_profile_values[0][i][1], angle_profile_values[0][i][2],
                                    trajectories[trajectory_index, i, 0], trajectories[trajectory_index, i, 1],
                                    trajectories[trajectory_index, i, 2]])
        row = row.reshape(1, -1)
        row = np.concatenate((row,  lidar[trajectory_index, i].reshape(1, -1)), axis=1)
        # print(row.shape)
        data = np.concatenate((data, row), axis=0)

data = data.reshape((-1,seq_len - 1,10+100*3))
print(f"Final dataset shape: {data.shape}")

Final dataset shape: (5, 19, 310)


In [63]:
np.save('data_lunar_mesh_ex.npy', data)
print(data.shape)
# Testing denormalizing and normalizing data
print(block_rss)
print(normalized)

from data_manipulations import denormalize
print(denormalize(normalized, min_rss, max_rss))


(5, 19, 310)
[-17.287508  -17.719418  -17.488415  -17.114822  -13.450251  -13.425502
 -13.517126  -13.392307  -16.128942  -15.822041  -15.74034   -16.171076
 -18.468897  -21.152021  -20.960848  -19.926876  -19.726341  -15.908092
 -14.7037945 -13.674521 ]
[0.8079166  0.8031175  0.8056842  0.8098353  0.8505528  0.8508278
 0.8498097  0.85119665 0.8207895  0.8241996  0.82510734 0.8203214
 0.79479    0.7649775  0.7671017  0.7785902  0.7808184  0.82324344
 0.8366245  0.84806085]
[-17.287506 -17.719421 -17.488419 -17.114822 -13.450249 -13.425499
 -13.517128 -13.392303 -16.128944 -15.822037 -15.740341 -16.171074
 -18.468895 -21.152023 -20.960846 -19.92688  -19.726341 -15.908089
 -14.703796 -13.674522]
